# 06. Model 2: CatBoost Classifier

**Employee Resignation Prediction Project**  
**Owner:** Member 2  
**Model Family:** Gradient Boosted Decision Trees (`CatBoostClassifier`)  
**Primary Selection Metric:** Average Precision (PR-AUC)  

### Methodology & Architectural Guardrails
1. **Strict Nested Outer-CV / Inner-Selection:** Every outer-validation fold is strictly prediction-only. Candidate selection, class-weight choice, and early stopping occur strictly within outer-training partitions.
2. **Centralized Configuration:** Model settings and candidate definitions are loaded and validated exclusively through `src.models.config` (`configs/models/catboost.yaml`). No hyperparameters are hardcoded or duplicated in this notebook.
3. **Native Categorical Support:** CatBoost natively splits and encodes categorical predictors (`Department`, `Gender`, `Job_Title`, `Education_Level`) without forced one-hot expansion or arbitrary integer encoding.
4. **Holdout Isolation:** The formal holdout partition (20,000 rows) is strictly isolated and never accessed for training, tuning, or diagnostic purposes.
5. **Refit Protocol:** After optimal candidate and iteration count selection on inner validation, a fresh model is refit on 100% of outer-training rows before evaluating on outer validation.

## Section A: Setup, Reproducibility & Centralized Configuration Validation

- Initialize environment, resolve repository root, and configure reproducible random seeds.
- Load `configs/models/catboost.yaml` using centralized loader `src.models.config.load_model_config`.
- Validate against the strict target schema (`strict_schema=True`).

In [1]:
import sys
import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Resolve repository root and initialize import path
PROJECT_ROOT = Path("../..").resolve()
if not (PROJECT_ROOT / "configs" / "feature_contract.yaml").exists():
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "configs" / "feature_contract.yaml").exists():
            PROJECT_ROOT = candidate.resolve()
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.config import load_model_config, validate_model_config, resolve_candidate
from src.models.catboost_model import CatBoostAdapter
from src.models.runner import run_5fold_cv
from src.evaluation.metrics import calculate_binary_metrics, probabilities_to_predictions

pd.set_option("display.max_columns", None)
CONFIG_PATH = PROJECT_ROOT / "configs" / "models" / "catboost.yaml"
FEATURE_CONTRACT_PATH = PROJECT_ROOT / "configs" / "feature_contract.yaml"
SPLIT_REGISTRY_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "models" / "catboost"

print(f"Project Root:          {PROJECT_ROOT}")
print(f"CatBoost Config Path:  {CONFIG_PATH}")
print(f"Feature Contract Path: {FEATURE_CONTRACT_PATH}")
print(f"Split Registry Path:   {SPLIT_REGISTRY_PATH}")

Project Root:          D:\Desktop\Employee_Resignation_Prediction
CatBoost Config Path:  D:\Desktop\Employee_Resignation_Prediction\configs\models\catboost.yaml
Feature Contract Path: D:\Desktop\Employee_Resignation_Prediction\configs\feature_contract.yaml
Split Registry Path:   D:\Desktop\Employee_Resignation_Prediction\data\processed\split_registry.csv


In [2]:
# Load and validate CatBoost configuration using centralized config utilities
catboost_config = load_model_config(CONFIG_PATH, strict_schema=True)
validate_model_config(catboost_config, strict_schema=True, model_id="catboost")

print(f"SUCCESS: CatBoost configuration loaded and strictly validated (schema v{catboost_config['model']['config_version']}):")
print(f"  Model ID:               {catboost_config['model']['id']}")
print(f"  Model Name:             {catboost_config['model']['name']}")
print(f"  Owner:                  {catboost_config['model']['owner']}")
print(f"  Algorithm Family:       {catboost_config['model']['algorithm_family']}")
print(f"  Implementation:         {catboost_config['model']['implementation']}")
print(f"  Random Seed:            {catboost_config['reproducibility']['random_seed']}")
print(f"  Inner Val Fraction:     {catboost_config['inner_validation']['fraction']}")
print(f"  Inner Early Stopping:   {catboost_config['inner_validation']['early_stopping_rounds']} rounds")
print(f"  Refit Outer Training:   {catboost_config['inner_validation']['refit_outer_train']}")
print(f"  Fixed Estimator Params: {catboost_config['fixed_params']}")

SUCCESS: CatBoost configuration loaded and strictly validated (schema v2.0.0):
  Model ID:               catboost
  Model Name:             CatBoost Classifier
  Owner:                  member_2
  Algorithm Family:       Gradient Boosted Decision Trees
  Implementation:         catboost.CatBoostClassifier
  Random Seed:            42
  Inner Val Fraction:     0.15
  Inner Early Stopping:   50 rounds
  Refit Outer Training:   True
  Fixed Estimator Params: {'loss_function': 'Logloss', 'eval_metric': 'PRAUC', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False}


## Section B: Data & Feature-Contract Validation

- Load `split_registry.csv` and filter strictly to the 80,000 `development` records across 5 frozen outer folds.
- Verify that the formal holdout partition (20,000 rows) is isolated and never accessed.
- Load `feature_contract.yaml` and verify all 17 approved predictors (4 categorical, 13 numerical) and target `Resigned`.
- Verify CatBoost native categorical handling without one-hot encoding or manual scaling.

In [3]:
# 1. Load Split Registry and enforce holdout isolation
split_registry = pd.read_csv(SPLIT_REGISTRY_PATH)
dev_registry = split_registry[split_registry["partition"] == "development"].copy()
holdout_registry = split_registry[split_registry["partition"] == "holdout"].copy()

assert len(dev_registry) == 80000, f"Expected 80,000 dev rows, found {len(dev_registry)}"
assert len(holdout_registry) == 20000, f"Expected 20,000 holdout rows, found {len(holdout_registry)}"
assert set(dev_registry["cv_fold"].unique()) == {1, 2, 3, 4, 5}, "Unexpected outer folds in dev registry"

# 2. Load Raw Dataset and filter strictly to development rows
raw_data = pd.read_csv(RAW_DATA_PATH)
assert len(raw_data) == 100000, f"Expected 100,000 raw rows, found {len(raw_data)}"

# Strictly restrict all work to development partition
dev_data = raw_data[raw_data["Employee_ID"].isin(dev_registry["Employee_ID"])].copy().reset_index(drop=True)
assert len(dev_data) == 80000, f"Expected 80,000 dev rows, found {len(dev_data)}"

# Formal Holdout Isolation Guard: Verify 0% overlap
assert not dev_data["Employee_ID"].isin(holdout_registry["Employee_ID"]).any(), "FATAL: Formal holdout contamination detected!"
print("Holdout Isolation Guard: VERIFIED (0 holdout records present in development working set).")

# Development target distribution
target_counts = dev_data["Resigned"].value_counts()
prevalence = dev_data["Resigned"].mean()
print(f"Development Target Distribution:\n{target_counts}")
print(f"Development Positive Class Prevalence: {prevalence:.4%} (True={target_counts.get(True, target_counts.get(1, 0))})")

Holdout Isolation Guard: VERIFIED (0 holdout records present in development working set).
Development Target Distribution:
Resigned
False    71992
True      8008
Name: count, dtype: int64
Development Positive Class Prevalence: 10.0100% (True=8008)


In [4]:
# Load and validate feature contract
with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as f:
    feature_contract = yaml.safe_load(f)

categorical_cols = [f["name"] for f in feature_contract["categorical_features"]]
numerical_cols = [f["name"] for f in feature_contract["numerical_features"]]
all_predictors = categorical_cols + numerical_cols
target_col = feature_contract["target"]["column"]

assert len(all_predictors) == 17, f"Expected 17 predictors, found {len(all_predictors)}"
assert len(categorical_cols) == 4, f"Expected 4 categorical predictors, found {len(categorical_cols)}"
assert len(numerical_cols) == 13, f"Expected 13 numerical predictors, found {len(numerical_cols)}"
assert target_col == "Resigned", f"Expected target 'Resigned', found {target_col}"

# Verify all columns exist in development data
missing_cols = set(all_predictors + [target_col]) - set(dev_data.columns)
assert not missing_cols, f"Missing columns in dev_data: {missing_cols}"

print(f"Feature Contract Verified:")
print(f"  Categorical Predictors ({len(categorical_cols)}): {categorical_cols}")
print(f"  Numerical Predictors   ({len(numerical_cols)}): {numerical_cols}")
print(f"  Target Column:                {target_col}")

Feature Contract Verified:
  Categorical Predictors (4): ['Department', 'Gender', 'Job_Title', 'Education_Level']
  Numerical Predictors   (13): ['Age', 'Years_At_Company', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score']
  Target Column:                Resigned


## Section C: Frozen Candidate Registry (C01–C08)

- Displays the approved, frozen candidate registry loaded directly from `configs/models/catboost.yaml`.
- The 8 discrete configurations provide systematic ablation across:
  - **Tree Depth:** 4 (conservative), 6 (baseline), 8 (high capacity)
  - **Learning Rate:** 0.03 (conservative), 0.05 (baseline), 0.10 (fast)
  - **L2 Leaf Regularization:** 3, 5, 10
  - **Class Imbalance Strategy:** Unweighted (`null`) vs Weighted (`"Balanced"`)
- All candidates use a bounded iteration ceiling of 600 with inner-validation early stopping (`eval_metric="PRAUC"`, patience 50).

In [5]:
# Display the frozen candidate registry directly from the validated YAML configuration
candidates_list = catboost_config.get("candidates", [])
candidates_records = []
for cand in candidates_list:
    cid = cand["id"]
    params = cand.get("params", {})
    candidates_records.append({
        "Candidate ID": cid,
        "Description": cand.get("description", ""),
        "Iterations": params.get("iterations", 600),
        "Depth": params.get("depth"),
        "Learning Rate": params.get("learning_rate"),
        "L2 Leaf Reg": params.get("l2_leaf_reg"),
        "Random Strength": params.get("random_strength", "default (1)"),
        "Class Weighting": params.get("auto_class_weights", "null (unweighted)"),
    })

candidates_df = pd.DataFrame(candidates_records)
print(f"CatBoost Frozen Candidate Registry (Total Configurations: {len(candidates_df)}):\n")
display(candidates_df)

CatBoost Frozen Candidate Registry (Total Configurations: 8):



,Candidate ID,Description,Iterations,Depth,Learning Rate,L2 Leaf Reg,Random Strength,Class Weighting
0,C01,"Baseline unweighted (depth=6, lr=0.05, l2=3)",600,6,0.05,3,default (1),NaN
1,C02,"Baseline Balanced (depth=6, lr=0.05, l2=3)",600,6,0.05,3,default (1),Balanced
2,C03,"Conservative / Shallow unweighted (depth=4, lr...",600,4,0.03,5,default (1),NaN
3,C04,"Conservative / Shallow Balanced (depth=4, lr=0...",600,4,0.03,5,default (1),Balanced
4,C05,"High capacity unweighted (depth=8, lr=0.05, l2=5)",600,8,0.05,5,default (1),NaN
5,C06,"High capacity Balanced (depth=8, lr=0.05, l2=5)",600,8,0.05,5,default (1),Balanced
6,C07,"Regularized low-rate Balanced (depth=6, lr=0.0...",600,6,0.03,10,1,Balanced
7,C08,"Fast learning Balanced (depth=6, lr=0.10, l2=5)",600,6,0.10,5,default (1),Balanced


## Section D: Lightweight Smoke & Integration Validation

Before executing formal 5-fold cross-validation on the 80,000 development records, this section performs a lightweight integration smoke test to verify:
1. End-to-end compatibility of `CatBoostAdapter` with `src.models.runner.run_5fold_cv`.
2. Proper stratified inner-split partitioning on outer-training rows.
3. Early stopping with PR-AUC metric capturing `selected_tree_count_`.
4. Final model refitting on 100% of outer-training rows with the fixed selected tree count.
5. Preservation of native categorical predictors without encoding errors.
6. Output probability bounds strictly in $[0.0, 1.0]$.
7. **Zero artifact side-effects:** No files in `results/models/catboost` are created or modified.

In [6]:
# Lightweight smoke validation on a 200-record development slice across 5 mock folds
print("Executing lightweight CatBoost integration smoke test...")
smoke_pos = dev_data[dev_data["Resigned"] == 1].sample(n=50, random_state=42)
smoke_neg = dev_data[dev_data["Resigned"] == 0].sample(n=150, random_state=42)
smoke_sample = pd.concat([smoke_pos, smoke_neg], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

smoke_registry = pd.DataFrame({
    "Employee_ID": smoke_sample["Employee_ID"],
    "partition": "development",
    "cv_fold": np.tile([1, 2, 3, 4, 5], 40),
})

# Resolve candidate C01 for smoke testing with low iterations for fast execution
smoke_resolved = resolve_candidate(catboost_config, candidate_id="C01")
smoke_resolved["params"]["iterations"] = 10
smoke_resolved["params"]["verbose"] = False

def smoke_adapter_factory():
    return CatBoostAdapter(
        config=smoke_resolved,
        categorical_features=categorical_cols,
        inner_val_fraction=0.20,
        early_stopping_rounds=5,
        refit_outer_train=True,
    )

smoke_start = time.time()
smoke_adapters = []

def smoke_adapter_factory():
    ad = CatBoostAdapter(
        config=smoke_resolved,
        categorical_features=categorical_cols,
        inner_val_fraction=0.20,
        early_stopping_rounds=5,
        refit_outer_train=True,
    )
    smoke_adapters.append(ad)
    return ad

smoke_result = run_5fold_cv(
    adapter_factory=smoke_adapter_factory,
    data=smoke_sample,
    split_registry=smoke_registry,
    feature_cols=all_predictors,
    target_col="Resigned",
    id_col="Employee_ID",
    fold_col="cv_fold",
    partition_col="partition",
    expected_model_name="CatBoost Classifier",
)
smoke_duration = time.time() - smoke_start

smoke_preds = smoke_result.predictions
assert len(smoke_preds) == 200, f"Expected 200 predictions, found {len(smoke_preds)}"
assert not smoke_preds["y_probability"].isna().any(), "NaN probabilities in smoke test!"
assert (smoke_preds["y_probability"] >= 0.0).all() and (smoke_preds["y_probability"] <= 1.0).all(), "Probabilities outside [0, 1]!"

# Verify selection evidence API across smoke adapters
assert len(smoke_adapters) == 5, f"Expected 5 smoke adapters, found {len(smoke_adapters)}"
smoke_evidence = smoke_adapters[-1].get_selection_evidence()
assert "winning_candidate_id" in smoke_evidence
assert smoke_evidence["selected_tree_count"] is not None
assert smoke_evidence["candidate_selection_time_seconds"] >= 0.0
assert smoke_evidence["refit_time_seconds"] >= 0.0

print(f"Smoke test PASSED in {smoke_duration:.2f}s:")
print(f"  Predictions shape:       {smoke_preds.shape}")
print(f"  Probability range:       [{smoke_preds['y_probability'].min():.4f}, {smoke_preds['y_probability'].max():.4f}]")
print(f"  Smoke Pooled AP (PR-AUC): {smoke_result.summary_metrics['pooled_average_precision']:.4f}")
print(f"  Smoke Pooled ROC-AUC:    {smoke_result.summary_metrics['pooled_roc_auc']:.4f}")
print(f"  Smoke Selection Evidence: {smoke_evidence['selected_tree_count']} trees in {smoke_evidence['candidate_selection_time_seconds']:.4f}s (refit: {smoke_evidence['refit_time_seconds']:.4f}s)")

Executing lightweight CatBoost integration smoke test...


Smoke test PASSED in 4.75s:
  Predictions shape:       (200, 6)
  Probability range:       [0.4602, 0.5053]
  Smoke Pooled AP (PR-AUC): 0.2207
  Smoke Pooled ROC-AUC:    0.4497
  Smoke Selection Evidence: 3 trees in 0.2655s (refit: 0.1015s)


## Section E: Formal Nested Five-Fold Cross-Validation (PRESENT BUT NOT EXECUTED)

### Execution Control Gate
- **`RUN_FORMAL_CV = False` is enforced for Step 2B.**
- In Step 3 (formal evaluation), toggling `RUN_FORMAL_CV = True` will execute:
  - 5 outer cross-validation folds on all 80,000 development records.
  - `candidate_id="auto"`: evaluates candidates C01–C08 on 15% inner-validation split derived strictly from outer-training rows.
  - Dynamic selection of best candidate by inner-validation Average Precision (PR-AUC).
  - Capture of optimal tree count from PR-AUC early stopping.
  - Refit of a fresh `CatBoostClassifier` on 100% of outer-training rows (64,000 records) using the selected fixed tree count.
  - Out-of-fold prediction generation for the 16,000 outer-validation rows.
  - Retention of all 5 fitted adapter instances via factory closure to prevent metadata loss.

In [7]:
# ==============================================================================
# FORMAL EXECUTION GATE
# Set to False during Step 2B to prevent premature formal CV execution.
# Will be toggled to True in Step 3 for the formal benchmark.
# ==============================================================================
RUN_FORMAL_CV = False

if RUN_FORMAL_CV:
    print("Beginning Formal Nested 5-Fold Cross-Validation on 80,000 Development Records...")
    formal_start = time.time()
    
    fitted_adapters = []
    
    def formal_adapter_factory():
        adapter = CatBoostAdapter(
            config=catboost_config,
            candidate_id="auto",  # Inner-validation candidate selection across C01-C08
            categorical_features=categorical_cols,
        )
        fitted_adapters.append(adapter)
        return adapter
    
    oof_result = run_5fold_cv(
        adapter_factory=formal_adapter_factory,
        data=dev_data,
        split_registry=split_registry,
        feature_cols=all_predictors,
        target_col=target_col,
        id_col="Employee_ID",
        fold_col="cv_fold",
        partition_col="partition",
        expected_model_name="CatBoost Classifier",
    )
    
    formal_duration = time.time() - formal_start
    print(f"Formal 5-Fold CV completed in {formal_duration:.1f}s ({formal_duration / 60:.2f} min).")
    
    # Rigorous post-execution verification
    assert len(fitted_adapters) == 5, f"Expected exactly 5 fitted adapters, found {len(fitted_adapters)}"
    assert len(oof_result.fold_metrics) == 5, f"Expected 5 fold metrics rows, found {len(oof_result.fold_metrics)}"
    expected_folds = [1, 2, 3, 4, 5]
    assert list(oof_result.fold_metrics["fold"]) == expected_folds, f"Unexpected fold IDs: {list(oof_result.fold_metrics['fold'])}"
else:
    print("Formal 5-fold cross-validation is intentionally SKIPPED in Step 2B.")
    print("Set RUN_FORMAL_CV = True in Step 3 to execute the formal experiment.")
    oof_result = None
    fitted_adapters = []

Formal 5-fold cross-validation is intentionally SKIPPED in Step 2B.
Set RUN_FORMAL_CV = True in Step 3 to execute the formal experiment.


## Section F: Formal Evaluation & Artifact Persistence (PRESENT BUT NOT EXECUTED)

- Guarded by `if oof_result is not None:`.
- In Step 2B, this block is inactive; no files are generated or modified in `results/models/catboost`.
- In Step 3, this block persists four canonical artifacts:
  1. `candidate_selection.csv`: Complete per-fold selection history across C01–C08, winning candidate, selected trees, class-weighting, and granular runtimes.
  2. `oof_predictions.csv`: 80,000 out-of-fold probability predictions including `model_id="catboost"`.
  3. `fold_metrics.csv`: Per-fold evaluation metrics (PR-AUC, ROC-AUC, Brier score, F1, Balanced Accuracy, fit/predict times).
  4. `summary_metrics.json`: Aggregated mean/std and pooled summary metrics.
  5. `catboost_run_metadata.json`: Comprehensive execution provenance, environment versions, live file SHA-256 hashes, and confusion matrix.

In [8]:
# ==============================================================================
# METRICS DISPLAY & ARTIFACT PERSISTENCE (ACTIVE ONLY AFTER FORMAL CV)
# ==============================================================================
if oof_result is not None:
    print("\n--- Out-of-Fold Fold-Level Performance ---")
    display(oof_result.fold_metrics)
    
    print("\n--- Summary Performance Metrics ---")
    summary_df = pd.DataFrame([oof_result.summary_metrics])
    display(summary_df)
    
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. Candidate Selection Evidence Artifact (candidate_selection.csv)
    candidate_records = []
    for fold_num, adapter in zip([1, 2, 3, 4, 5], fitted_adapters):
        evidence = adapter.get_selection_evidence()
        fold_metric_row = oof_result.fold_metrics[oof_result.fold_metrics["fold"] == fold_num].iloc[0]
        
        record = {
            "outer_fold": fold_num,
            "outer_train_rows": 64000,
            "outer_validation_rows": 16000,
            "inner_train_rows": evidence.get("inner_train_rows"),
            "inner_validation_rows": evidence.get("inner_val_rows"),
            "winning_candidate": evidence.get("winning_candidate_id"),
            "winning_inner_ap": evidence.get("winning_inner_ap"),
            "selected_tree_count": evidence.get("selected_tree_count"),
            "selected_class_weight": str(evidence.get("selected_class_weight")),
            "random_seed": evidence.get("random_seed"),
            "inner_validation_fraction": evidence.get("inner_val_fraction"),
            "candidate_selection_time_seconds": evidence.get("candidate_selection_time_seconds"),
            "refit_time_seconds": evidence.get("refit_time_seconds"),
            "predict_time_seconds": float(fold_metric_row["predict_time_seconds"]),
        }
        
        # Include per-candidate inner AP and selected tree count
        all_cands = evidence.get("all_candidates", {})
        for c_id in [f"C{i:02d}" for i in range(1, 9)]:
            c_info = all_cands.get(c_id, {})
            record[f"{c_id}_inner_ap"] = c_info.get("ap_score")
            record[f"{c_id}_selected_trees"] = c_info.get("selected_trees")
            
        candidate_records.append(record)
        
    candidate_selection_df = pd.DataFrame(candidate_records)
    candidate_selection_path = RESULTS_DIR / "candidate_selection.csv"
    candidate_selection_df.to_csv(candidate_selection_path, index=False)
    print(f"Saved: {candidate_selection_path}")
    
    # 2. OOF Predictions Artifact (with model_id)
    oof_export_df = oof_result.predictions.copy()
    oof_export_df["model_id"] = "catboost"
    cols = ["Employee_ID", "cv_fold", "y_true", "y_probability", "y_pred_0_5", "model_name", "model_id"]
    oof_export_df = oof_export_df[cols]
    oof_preds_path = RESULTS_DIR / "oof_predictions.csv"
    oof_export_df.to_csv(oof_preds_path, index=False)
    print(f"Saved: {oof_preds_path}")
    
    # 3. Fold Metrics Artifact
    fold_metrics_path = RESULTS_DIR / "fold_metrics.csv"
    oof_result.fold_metrics.to_csv(fold_metrics_path, index=False)
    print(f"Saved: {fold_metrics_path}")
    
    # 4. Summary Metrics Artifact (summary_metrics.json)
    summary_metrics_path = RESULTS_DIR / "summary_metrics.json"
    with open(summary_metrics_path, "w", encoding="utf-8") as f:
        json.dump(oof_result.summary_metrics, f, indent=2)
    print(f"Saved: {summary_metrics_path}")
    
    # 5. Live Provenance & Comprehensive Run Metadata (catboost_run_metadata.json)
    import hashlib
    from datetime import datetime, timezone
    import catboost
    
    def get_sha256(filepath: Path) -> str:
        h = hashlib.sha256()
        with open(filepath, "rb") as f:
            while chunk := f.read(65536):
                h.update(chunk)
        return h.hexdigest()
        
    # Compute live hashes directly from on-disk canonical files
    dataset_sha = get_sha256(RAW_DATA_PATH)
    split_registry_sha = get_sha256(SPLIT_REGISTRY_PATH)
    feature_contract_sha = get_sha256(FEATURE_CONTRACT_PATH)
    catboost_yaml_sha = get_sha256(CONFIG_PATH)
    
    # Compute confusion matrix and standard binary metrics across all 80k OOF predictions
    from src.evaluation.metrics import calculate_binary_metrics
    pooled_binary_metrics = calculate_binary_metrics(
        y_true=oof_export_df["y_true"].to_numpy(),
        y_probability=oof_export_df["y_probability"].to_numpy(),
        y_pred=oof_export_df["y_pred_0_5"].to_numpy(),
        threshold=0.50,
    )
    
    run_metadata = {
        "model_id": "catboost",
        "model_name": "CatBoost Classifier",
        "catboost_version": catboost.__version__,
        "python_version": sys.version,
        "platform": sys.platform,
        "config_version": catboost_config["model"]["config_version"],
        "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "random_seed": catboost_config["reproducibility"]["random_seed"],
        "fixed_outer_folds": [1, 2, 3, 4, 5],
        "primary_selection_metric": "average_precision",
        "total_development_records": len(oof_export_df),
        "formal_holdout_records_used": 0,
        "total_formal_run_time_seconds": formal_duration,
        "provenance_hashes": {
            "dataset_sha256": dataset_sha,
            "split_registry_sha256": split_registry_sha,
            "feature_contract_sha256": feature_contract_sha,
            "catboost_yaml_sha256": catboost_yaml_sha,
        },
        "config_snapshot": catboost_config,
        "pooled_evaluation_metrics": pooled_binary_metrics,
        "summary_metrics": oof_result.summary_metrics,
        "fold_metrics": oof_result.fold_metrics.to_dict(orient="records"),
        "candidate_selection_summary": candidate_records,
    }
    
    run_metadata_path = RESULTS_DIR / "catboost_run_metadata.json"
    with open(run_metadata_path, "w", encoding="utf-8") as f:
        json.dump(run_metadata, f, indent=2)
    print(f"Saved: {run_metadata_path}")
    
    print("\nAll formal CatBoost artifacts serialized successfully.")
else:
    print("Formal evaluation skipped. No artifacts written to results/models/catboost/.")

Formal evaluation skipped. No artifacts written to results/models/catboost/.


## Section G: Verification & Step 2B Completion Summary

### Verification Checklist
- [x] Centralized configuration loading via `src.models.config.load_model_config`.
- [x] Strict schema validation enforced (`validate_model_config(..., strict_schema=True)`).
- [x] No hyperparameters hardcoded in notebook Python dictionaries.
- [x] Frozen candidate registry C01–C08 displayed dynamically from YAML.
- [x] Data loaded from canonical raw dataset and split registry.
- [x] Strict restriction to 80,000 development records.
- [x] Formal holdout partition (20,000 records) strictly isolated and unaccessed.
- [x] Feature contract verified (17 predictors: 4 categorical, 13 numerical; target: `Resigned`).
- [x] Lightweight integration smoke test passed with valid probabilities in $[0, 1]$.
- [x] Formal nested CV staged under execution control gate (`RUN_FORMAL_CV = False`).
- [x] No formal result artifacts generated in `results/models/catboost/`.
- [x] Ready for formal five-fold cross-validation in Step 3.